# LAB 10  K-Means Clustering
## Student Information
Name: **Nazakat Ali**  
Student ID: **023-22-0146**  
Section: **Buffer Batch**  
GitHub: **https://github.com/Nizakatkhannk**  
Kaggle: **https://www.kaggle.com/nizakatkhannk**  
Dataset: Mall Customers (Kaggle) + Wine (UCI)  
Dataset Source: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python and https://archive.ics.uci.edu/dataset/109/wine



## Part 0 — Setup and First Look

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
sns.set_style('whitegrid'); SEED = 42
np.set_printoptions(precision=3, suppress=True)
def say(t): display(Markdown(t))
PAL = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

### Task 0.1 — Load and check the data

In [ ]:
CSV = 'Mall_Customers.csv'
hits = glob.glob(f'**/{CSV}', recursive=True)
if hits: CSV = hits[0]
else:
    try:
        try: import kagglehub
        except ImportError:
            import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub'], check=True); import kagglehub
        CSV = os.path.join(kagglehub.dataset_download('vjchoudhary7/customer-segmentation-tutorial-in-python'), CSV)
    except Exception as e:
        print('Auto-download failed:', e, '\nPlease upload Mall_Customers.csv manually.')
        from google.colab import files; CSV = list(files.upload().keys())[0]

df = pd.read_csv(CSV).rename(columns={'Annual Income (k$)': 'Income', 'Spending Score (1-100)': 'Spending', 'Genre': 'Gender'})
print('Shape:', df.shape, '| missing values:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
print(df['Gender'].value_counts().to_string())
display(df.head()); display(df[['Age', 'Income', 'Spending']].describe().round(2))
X2 = df[['Income', 'Spending']].values.astype(float)       # the 200 x 2 matrix used in most tasks

### Task 0.2 — Look at the data before clustering

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for a, (x, y) in zip(ax, [('Income', 'Spending'), ('Age', 'Spending'), ('Age', 'Income')]):
    a.scatter(df[x], df[y], s=25, alpha=.75); a.set_xlabel(x); a.set_ylabel(y); a.set_title(f'{y} vs {x}')
plt.tight_layout(); plt.show()

### Answers — Part 0
**Groups by eye.** *Spending vs Income* is the only plot with clearly separated clumps: a central blob (mid income, mid spending) and four outer clumps (low income/low spending, low income/high spending, high income/low spending, high income/high spending) — **5 groups**. *Spending vs Age* shows a loose fan (young customers reach high spending scores, customers over ~40 rarely do) and *Income vs Age* is an unstructured cloud: neither shows distinct clumps (at best 2–3 overlapping regions). So the Income–Spending plot shows the groups most clearly, which is why most of the lab uses `X2 = [Income, Spending]`. *(Confirm this against your own three plots.)*

**Why no accuracy or train/test split?** There is no target column: nothing tells us the "true" group of a customer, so there is no answer to compare predictions with, and a held-out set would have nothing to be scored against. Quality has to be judged *internally* (inertia, silhouette — how tight and well-separated the clusters are), *visually*, and by *usefulness* (can a manager act on the segments?). Cluster numbers are also arbitrary names, so "accuracy against the label" would not even be well defined.

## Part 1 — K-Means From Scratch

### Task 1.1 — Lecture Example 1 in code (one feature)

In [ ]:
x = np.array([2, 4, 10, 12, 3, 20, 30, 11, 25], dtype=float)
m = np.array([4., 11.])                       # starting centroids
it = 0
while True:
    it += 1
    dist = np.abs(x[:, None] - m[None, :])    # (9, 2) distance matrix
    lab = dist.argmin(axis=1)                 # assign step
    new_m = np.array([x[lab == k].mean() if (lab == k).any() else m[k] for k in range(2)])   # update step
    print(f'Iteration {it}: C1 = {x[lab==0].astype(int).tolist()}  C2 = {x[lab==1].astype(int).tolist()}  -> new centroids {new_m}')
    if np.allclose(new_m, m): break           # no centroid moved -> converged
    m = new_m
inertia1 = ((x - m[lab]) ** 2).sum()
print(f'\nFinal centroids: {m} | inertia = {inertia1:.1f} | iterations = {it}')
print('Matches lecture (centroids 7 and 25, inertia 150):', np.allclose(m, [7, 25]) and np.isclose(inertia1, 150))

### Task 1.2 — Lecture Example 2 in code (two features)

In [ ]:
P = np.array([[185, 72], [170, 56], [168, 60], [179, 68], [182, 72], [188, 77]], dtype=float)
names = ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']
M = P[[0, 1]].copy(); it = 0; first_table = None
while True:
    it += 1
    D = np.sqrt(((P[:, None, :] - M[None, :, :]) ** 2).sum(axis=2))     # Euclidean, broadcasting -> (6, 2)
    lab = D.argmin(axis=1)
    if first_table is None: first_table = pd.DataFrame(D.round(2), index=names, columns=['d to M1', 'd to M2'])
    newM = np.array([P[lab == k].mean(axis=0) if (lab == k).any() else M[k] for k in range(2)])
    print(f'Iteration {it}: ' + ', '.join(f'{n}->C{l+1}' for n, l in zip(names, lab)) + f' | centroids: M1={newM[0].round(2)}, M2={newM[1].round(2)}')
    if np.allclose(newM, M): break
    M = newM
print('\nFirst-iteration distance table (start centroids P1, P2):'); display(first_table)
final_D = np.sqrt(((P[:, None, :] - M[None, :, :]) ** 2).sum(axis=2)); display(pd.DataFrame(final_D.round(2), index=names, columns=['d to M1 (final)', 'd to M2 (final)']))
by_hand = np.sqrt((168 - 170) ** 2 + (60 - 56) ** 2)
print(f'Hand check: d(P3, M2) = sqrt((168-170)^2 + (60-56)^2) = sqrt(4 + 16) = {by_hand:.2f}  | code gives {first_table.loc["P3","d to M2"]}  (lecture: 4.47)')

### Task 1.3 — Write your own K-Means function

In [ ]:
def my_kmeans(X, init_centroids, max_iter=100, verbose=False):
    # Lloyd's K-Means for any number of features. NO sklearn KMeans inside.
    X = np.asarray(X, dtype=float); C = np.asarray(init_centroids, dtype=float).copy()
    if X.ndim == 1: X = X[:, None]
    if C.ndim == 1: C = C[:, None]
    K = len(C); prev = None
    for it in range(1, max_iter + 1):
        d = np.sqrt(((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2))     # assign step (n, K)
        labels = d.argmin(axis=1)
        newC = np.array([X[labels == k].mean(axis=0) if (labels == k).any() else C[k] for k in range(K)])  # empty cluster -> keep old centroid
        if verbose: print(f'iter {it}:', {k: X[labels == k].ravel().tolist() for k in range(K)}, 'centroids', newC.ravel())
        converged = prev is not None and np.array_equal(labels, prev)      # no point changed cluster
        C, prev = newC, labels
        if converged: break
    inertia = ((X - C[labels]) ** 2).sum()
    return labels, C, inertia, it

lab_t, C_t, in_t, it_t = my_kmeans(x, [4, 11])
print('Example 1 -> centroids', C_t.ravel(), '| inertia', in_t, '| iterations', it_t, '| correct:', np.allclose(sorted(C_t.ravel()), [7, 25]) and np.isclose(in_t, 150))

### Task 1.4 — Check your function against scikit-learn

In [ ]:
init5 = X2[:5].copy()
l_me, C_me, in_me, it_me = my_kmeans(X2, init5, max_iter=300)
sk = KMeans(n_clusters=5, init=init5, n_init=1, tol=0, algorithm='lloyd', max_iter=300).fit(X2)
res = pd.DataFrame({'my_kmeans': [it_me, in_me], 'sklearn KMeans': [sk.n_iter_, sk.inertia_]}, index=['iterations', 'inertia']); display(res.round(4))
same_labels = np.array_equal(l_me, sk.labels_); cdiff = np.abs(C_me - sk.cluster_centers_).max()
print('Labels identical:', same_labels, '| max |centroid difference|:', cdiff, '| inertia equal:', np.isclose(in_me, sk.inertia_))
display(pd.DataFrame(np.c_[C_me, sk.cluster_centers_], columns=['my Income', 'my Spending', 'sk Income', 'sk Spending']).round(3))
say(f'''**Your numbers:** my function used **{it_me}** iterations vs sklearn's **{sk.n_iter_}**; inertia **{in_me:.3f}** vs **{sk.inertia_:.3f}**; labels identical: **{same_labels}**; largest centroid difference **{cdiff:.2e}**.''')

### Answers — Part 1
**Example 1, point 10.** After iteration 1 the clusters are C1 = {2, 3, 4} (centroid **3**) and C2 = {10, 11, 12, 20, 25, 30} (centroid **18**). At the start the centroids were 4 and 11, so 10 was only 1 away from C2's centroid 11 (vs 6 from 4). But C2 had absorbed the big values 20, 25 and 30, which pulled its centroid up to 18; now |10 − 3| = **7** is smaller than |10 − 18| = **8**, so 10 switches to C1. Final centroids are 7 and 25 with inertia 150, as in the lecture.

**What makes the loop stop?** It stops when an iteration changes no assignment (equivalently, no centroid moves). Stopping is guaranteed because each assign step and each update step can only *decrease or keep* the inertia (assigning to the nearest centroid cannot increase squared distance; the mean minimises the sum of squared distances within a cluster), the inertia is bounded below by 0, and there are only finitely many ways to split the points into K groups, so the algorithm cannot cycle forever. But it only guarantees a **local** minimum: a different start can end in a different, worse stable solution (Part 3 measures this).

**Did my function match scikit-learn?** See the printed comparison above. With identical starting centroids, `tol=0`, `algorithm='lloyd'` and `n_init=1`, both run the same deterministic procedure, so labels, centroids and inertia should agree exactly (up to ~1e-12 floating-point error) and iteration counts should match since I count iterations the same way (the final confirming pass is included). If anything differs, the usual reasons are: a different convergence test (sklearn's `tol` compares centroid movement; I compare labels), different tie-breaking when a point is equidistant to two centroids, a different empty-cluster rule, or floating-point summation order.

**What is a centroid, and why numeric features?** A centroid is the *average customer* of a cluster — a typical profile (e.g. income ≈ 87 k$, spending ≈ 82) that need not be a real customer. K-Means needs means and Euclidean distances, so every feature must be numeric (and meaningfully scaled); a category like "Gender" has no average or distance unless it is first encoded as numbers.

## Part 2 — Choosing K: Elbow and Silhouette

### Task 2.1 — Elbow and silhouette for K = 2 to 10

In [ ]:
Ks = list(range(2, 11)); rows = []
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X2)
    rows.append({'K': k, 'Inertia': km.inertia_, 'Silhouette': silhouette_score(X2, km.labels_)})
kt = pd.DataFrame(rows).set_index('K'); display(kt.round(4))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(kt.index, kt['Inertia'], 'o-'); ax[0].set_xlabel('K'); ax[0].set_ylabel('Inertia'); ax[0].set_title('Elbow curve')
ax[1].bar(kt.index, kt['Silhouette'], color='teal'); ax[1].set_xlabel('K'); ax[1].set_ylabel('Average silhouette'); ax[1].set_title('Silhouette by K')
plt.tight_layout(); plt.show()
bestK = int(kt['Silhouette'].idxmax()); print('K with the highest silhouette:', bestK)
drop = kt['Inertia'].diff().abs()
say(f'''**Your numbers:** highest silhouette at **K = {bestK}** ({kt.loc[bestK,'Silhouette']:.3f}). Inertia drops by **{drop[5]:.0f}** from K=4→5 ({100*drop[5]/kt.loc[4,'Inertia']:.0f}%),
but only by **{drop[6]:.0f}** from K=5→6 ({100*drop[6]/kt.loc[5,'Inertia']:.0f}%) and **{drop[7]:.0f}** from K=6→7.''')

### Task 2.2 — Silhouette by hand

In [ ]:
km5 = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2); lab5 = km5.labels_

def silhouette_point(X, labels, i):
    d = np.sqrt(((X - X[i]) ** 2).sum(axis=1))                  # distances from point i to every point
    own = labels[i]; same = (labels == own); same[i] = False    # exclude the point itself
    if same.sum() == 0: return 0.0, np.nan, 0.0                 # singleton cluster: s = 0 by convention
    a = d[same].mean()
    b = min(d[labels == k].mean() for k in np.unique(labels) if k != own)
    return a, b, (b - a) / max(a, b)

sil = silhouette_samples(X2, lab5)
a0, b0, s0 = silhouette_point(X2, lab5, 0)
print(f'Customer 1 (row 0): a = {a0:.3f}, b = {b0:.3f}, s = {s0:.4f}  | silhouette_samples gives {sil[0]:.4f}  | equal: {np.isclose(s0, sil[0])}')
all_equal = all(np.isclose(silhouette_point(X2, lab5, i)[2], sil[i]) for i in range(len(X2)))
print('My function matches silhouette_samples for ALL 200 customers:', all_equal)

tab = pd.DataFrame({'Cluster': lab5, 'sil': sil}).groupby('Cluster')['sil'].agg(Count='count', Mean='mean', Min='min')
display(tab.round(4)); neg = int((sil < 0).sum()); print('Customers with a negative silhouette:', neg)

fig, ax = plt.subplots(figsize=(8, 6)); y0 = 10
for k in range(5):
    s = np.sort(sil[lab5 == k]); ax.fill_betweenx(np.arange(y0, y0 + len(s)), 0, s, color=PAL[k], alpha=.8); ax.text(-0.08, y0 + len(s) / 2, f'C{k}'); y0 += len(s) + 10
ax.axvline(sil.mean(), color='red', ls='--', label=f'average = {sil.mean():.3f}'); ax.set_xlabel('Silhouette value'); ax.set_yticks([]); ax.set_title('Silhouette plot (K = 5)'); ax.legend(); plt.show()
weak = tab['Mean'].idxmin()
say(f'''**Your numbers:** customer 1 has **a = {a0:.2f}, b = {b0:.2f}, s = {s0:.3f}**. The weakest cluster is **C{weak}** (mean s = {tab.loc[weak,'Mean']:.3f}, minimum {tab.loc[weak,'Min']:.3f}); **{neg}** customers have a negative silhouette.''')

### Task 2.3 — The lecture practice problem

In [ ]:
pts = np.array([1, 3, 5, 7, 11, 13, 15], dtype=float)
lab_p, C_p, in_p, it_p = my_kmeans(pts, [1, 11], verbose=True)
print(f'\nFinal clusters: C1 = {pts[lab_p==0].astype(int).tolist()}, C2 = {pts[lab_p==1].astype(int).tolist()} | centroids {C_p.ravel()} | iterations {it_p} | inertia {in_p:.1f}')
Xp = pts[:, None]; sp = silhouette_samples(Xp, lab_p); rows = []
for v in [1, 7, 11]:
    i = int(np.where(pts == v)[0][0]); a, b, s = silhouette_point(Xp, lab_p, i); rows.append({'point': v, 'a': a, 'b': b, 's (mine)': s, 's (sklearn)': sp[i]})
display(pd.DataFrame(rows).set_index('point').round(4))

*Paper work:* write the iterations of this practice problem by hand (clusters and centroids after each iteration — the verbose printout above is the answer key) and keep the photo/notes for revision.

### Answers — Part 2
**Elbow vs silhouette.** Read the printed numbers from Task 2.1: the inertia falls steeply up to K = 5 and then the decrease per extra cluster becomes much smaller (compare the K=4→5 drop with the K=5→6 and K=6→7 drops in the generated sentence) — the elbow is at **K = 5**. The silhouette is also highest at the K printed (**K = 5** for the Mall data). The two methods agree, and both match the five clumps seen by eye in Part 0. When the two disagree, report both and decide using the business use of the segments.

**Customer 1.** Its silhouette (printed above) is positive and, when well above 0.5, it is much closer to the members of its own cluster than to the members of the nearest other cluster (a ≪ b), it sits well inside its cluster. The cluster with the weakest average silhouette, and the count of negative values, are in the generated sentence; in this data the middle blob (mid income / mid spending) is usually the weakest because its neighbours touch it on all four sides.

**Practice problem.** The final clusters are {1, 3, 5, 7} and {11, 13, 15}. Point 1 has a = 4, b = 12 → s ≈ 0.67; point 7 has a = 4, b = 6 → s ≈ 0.33; point 11 has a = 3, b = 7 → s ≈ 0.57. **Point 7 is less well placed**: its average distance to its own cluster (4) is not much smaller than to the other cluster (6) because it sits at the border between the two groups, while point 1 is deep inside its cluster (b is three times a).

**Why not choose K by lowest inertia?** Inertia always decreases as K increases (K = n gives inertia 0, every point its own cluster), so the lowest inertia always says "use more clusters" and says nothing about whether the extra clusters are real. K must be chosen where extra clusters stop paying off (elbow), where clusters are well separated (silhouette), and where the segments are interpretable.

## Part 3 — Starting Points and Stability

### Task 3.1 — 30 different seeds

In [ ]:
settings = {'random, n_init=1': dict(init='random', n_init=1), 'k-means++, n_init=1': dict(init='k-means++', n_init=1), 'k-means++, n_init=10': dict(init='k-means++', n_init=10)}
runs = {}
for name, kw in settings.items():
    runs[name] = [KMeans(n_clusters=5, random_state=s, **kw).fit(X2) for s in range(30)]
inert = {n: np.array([r.inertia_ for r in rs]) for n, rs in runs.items()}
best_all = min(v.min() for v in inert.values()).round(1)
rows = []
for n, v in inert.items():
    r1 = v.round(1); rows.append({'Setting': n, 'Distinct inertia values': len(np.unique(r1)), 'Lowest': r1.min(), 'Highest': r1.max(),
                                 '% runs reaching best': 100 * np.mean(r1 == best_all)})
stab = pd.DataFrame(rows).set_index('Setting'); display(stab.round(2))
lo, hi = min(v.min() for v in inert.values()), max(v.max() for v in inert.values()); pad = max((hi - lo) * .05, 1)
plt.figure(figsize=(9, 4.5)); plt.hist([inert[n] for n in settings], bins=np.linspace(lo - pad, hi + pad, 16), label=list(settings), color=PAL[:3])
plt.xlabel('Final inertia'); plt.ylabel('Number of runs (out of 30)'); plt.title('Final inertia over 30 seeds (K = 5)'); plt.legend(); plt.show()

### Task 3.2 — Best run vs worst run

In [ ]:
rr = runs['random, n_init=1']; best_r = min(rr, key=lambda r: r.inertia_); worst_r = max(rr, key=lambda r: r.inertia_)
ari_bw = adjusted_rand_score(best_r.labels_, worst_r.labels_)
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for a, r, t in zip(ax, [best_r, worst_r], ['Best random-init run', 'Worst random-init run']):
    for k in range(5): a.scatter(*X2[r.labels_ == k].T, s=22, color=PAL[k], alpha=.8)
    a.scatter(*r.cluster_centers_.T, marker='X', s=220, c='black', edgecolor='white'); a.set_title(f'{t}\ninertia = {r.inertia_:.1f}'); a.set_xlabel('Income'); a.set_ylabel('Spending')
plt.tight_layout(); plt.show()
print(f'Best inertia {best_r.inertia_:.1f} | worst inertia {worst_r.inertia_:.1f} | ARI between them = {ari_bw:.3f}')
r0 = stab.loc['random, n_init=1']; r2 = stab.loc['k-means++, n_init=1']; r3 = stab.loc['k-means++, n_init=10']
say(f'''**Your numbers:** random starts gave **{int(r0['Distinct inertia values'])}** distinct inertia values and only **{r0['% runs reaching best']:.0f}%** of runs reached the best inertia ({best_all});
k-means++ (n_init=1): **{int(r2['Distinct inertia values'])}** values, **{r2['% runs reaching best']:.0f}%** best; k-means++ with n_init=10: **{int(r3['Distinct inertia values'])}** values, **{r3['% runs reaching best']:.0f}%** best.
Best vs worst random run: inertia {best_r.inertia_:.1f} vs {worst_r.inertia_:.1f}, ARI = {ari_bw:.3f}.''')

### Answers — Part 3
**Numbers.** The generated sentence above gives the count of distinct inertia values and the percentage of runs reaching the best inertia for each setting. Random initial centroids usually produce several different final inertia values (some runs land in a worse local minimum), and only a fraction of the 30 runs hit the best one. k-means++ with a single start already reaches the best inertia far more often, and **k-means++ with `n_init=10` should reach it in (nearly) all 30 runs**, because it keeps the best of ten starts.

**Why can the same data and K give different answers?** The assign/update loop is a descent that stops at the *first* stable solution it reaches (a local minimum of the inertia). Which one that is depends on the starting centroids; the problem has many local minima, so different starts can converge to different clusterings. A bad start — e.g. two initial centroids inside one natural clump and none in another — typically leaves one real group split in two and two others merged (visible in the best-vs-worst picture).

**How k-means++ and `n_init` help.** k-means++ picks the starting centroids one by one, preferring points far from the ones already chosen, so the initial centroids are spread across the data and a "two centroids in one clump" start is unlikely. `n_init=10` repeats the whole algorithm from 10 different starts and keeps the run with the **lowest inertia**, so even if a few starts are unlucky the best one is kept (see the last row of the table). The cost is 10× the computation, which is trivial for 200 points.

**Why ARI instead of comparing label numbers?** Cluster numbers are arbitrary: "cluster 2" in one run may be exactly the group that is "cluster 4" in another, so comparing label numbers would report disagreement for identical groupings. The Adjusted Rand Index compares *which pairs of points are together*, so it is invariant to renaming (1 = same grouping, ≈ 0 = agreement no better than chance).

## Part 4 — Scaling: Units Decide the Clusters

### Task 4.1 — Change the unit, change the clusters

In [ ]:
Xa = df[['Age', 'Income']].values.astype(float)            # Income in k$
Xd = Xa.copy(); Xd[:, 1] *= 1000                           # Income in dollars
def km_labels(X): return KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(X).labels_
L = {'raw k$': km_labels(Xa), 'raw dollars': km_labels(Xd), 'scaled k$': km_labels(StandardScaler().fit_transform(Xa)), 'scaled dollars': km_labels(StandardScaler().fit_transform(Xd))}
pairs = [('raw k$', 'raw dollars'), ('scaled k$', 'scaled dollars'), ('raw dollars', 'scaled dollars')]
ari_t = pd.DataFrame([{'Pair': f'{a} vs {b}', 'ARI': adjusted_rand_score(L[a], L[b])} for a, b in pairs]).set_index('Pair'); display(ari_t.round(4))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True, sharey=True)
for a, key in zip(ax, ['raw k$', 'raw dollars', 'scaled k$']):
    for k in range(4): a.scatter(Xa[L[key] == k, 0], Xa[L[key] == k, 1], s=22, color=PAL[k], alpha=.8)
    a.set_title(f'Clusters from: {key}'); a.set_xlabel('Age'); a.set_ylabel('Income (k$)')
plt.tight_layout(); plt.show()
say(f'''**Your numbers:** raw k$ vs raw dollars ARI = **{ari_t.iloc[0,0]:.3f}**; scaled k$ vs scaled dollars ARI = **{ari_t.iloc[1,0]:.3f}**; raw dollars vs scaled dollars ARI = **{ari_t.iloc[2,0]:.3f}**.''')

### Task 4.2 — Wine data: raw vs scaled

In [ ]:
wine = load_wine(); Xw, yw = wine.data, wine.target; fn = np.array(wine.feature_names)
sd = pd.Series(Xw.std(axis=0), index=fn).sort_values(ascending=False)
print('Three largest std:'); print(sd.head(3).round(3).to_string()); print('\nSmallest std:'); print(sd.tail(1).round(4).to_string())
Xw_s = StandardScaler().fit_transform(Xw)
kw_raw = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(Xw); kw_sc = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(Xw_s)
wres = pd.DataFrame({'ARI vs true cultivar': [adjusted_rand_score(yw, kw_raw.labels_), adjusted_rand_score(yw, kw_sc.labels_)],
                     'Silhouette (own space)': [silhouette_score(Xw, kw_raw.labels_), silhouette_score(Xw_s, kw_sc.labels_)],
                     'Cluster sizes': [np.bincount(kw_raw.labels_).tolist(), np.bincount(kw_sc.labels_).tolist()]}, index=['raw', 'scaled'])
display(wres)

### Task 4.3 — Draw it

In [ ]:
Z = PCA(n_components=2, random_state=SEED).fit_transform(Xw_s)       # one PCA of the scaled data, used only for drawing
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True, sharey=True)
for a, lab_, t in zip(ax, [yw, kw_raw.labels_, kw_sc.labels_], ['True cultivars', 'K-Means on RAW data', 'K-Means on SCALED data']):
    for k in range(3): a.scatter(*Z[lab_ == k].T, s=28, color=PAL[k], alpha=.8)
    a.set_title(t); a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.tight_layout(); plt.show()
dom = sd.index[0]
say(f'''**Your numbers:** the dominant raw feature is **{dom}** (std = {sd.iloc[0]:.1f}) — about **{sd.iloc[0]/sd.iloc[1]:.0f}×** the next largest ({sd.index[1]}, {sd.iloc[1]:.1f}) and **{sd.iloc[0]/sd.iloc[-1]:,.0f}×** the smallest ({sd.index[-1]}, {sd.iloc[-1]:.3f}).
ARI vs true cultivar: raw **{wres.iloc[0,0]:.3f}**, scaled **{wres.iloc[1,0]:.3f}**. Silhouette: raw {wres.iloc[0,1]:.3f}, scaled {wres.iloc[1,1]:.3f}.''')

### Answers — Part 4
**Changing only the unit of Income.** Euclidean distance is √((ΔAge)² + (ΔIncome)²). With Income in k$ the two terms are of similar size (Age std ≈ 14 years, Income std ≈ 26 k$), so both features influence the clusters (though Income somewhat more). Multiplying Income by 1000 makes ΔIncome tens of thousands while ΔAge stays in the tens, so the Age term becomes negligible and **Income alone decides the clusters**: the groups become four horizontal income bands. The ARI between raw k$ and raw dollars (above) is therefore below 1 (see the printed value) even though the customers did not change — only the unit did. After `StandardScaler` both units give the same z-scores, so scaled k$ vs scaled dollars has ARI = 1.0: scaling removes the dependence on the unit.

**Wine.** The generated sentence names the feature that dominates the raw distance (`proline`, with a std of a few hundred) and compares its std with the others (`magnesium` ≈ 14, most features < 2, the smallest ≈ 0.1). Raw K-Means is therefore essentially clustering on proline alone and its ARI against the true cultivars is low; after scaling, all 13 chemical measurements contribute and the ARI is much higher (compare the two ARI values printed).

**Why the silhouette misleads.** The raw silhouette is higher although the raw clustering is worse. Silhouette measures how well-separated clusters are *in the space in which they were computed*. In the raw space, nearly all the distance comes from one feature (proline), so the data look like three tidy bands along one axis and the silhouette is high — but those bands are not the cultivars. In the scaled space, 13 features contribute, the cultivars overlap more in 13-D, so the silhouette is lower even though the grouping matches the truth better. The two silhouettes are measured in **different geometries** (different distance definitions), so comparing them across raw and scaled data is not a fair comparison; silhouette can only rank solutions *within the same feature space*.

**My rule for scaling in a real project (no labels).** Scale whenever the features have different units or very different numeric ranges and I have no deliberate reason to give some features more weight (the default for almost all tabular data); skip it only when all features share the same unit and the raw magnitudes are themselves meaningful (e.g. pixel intensities or coordinates in metres), and always check whether the segments change when I rescale a feature — if they do, the result depends on the unit and not on the customers.

## Part 5 — Outliers and Anomalies

### Task 5.1 — One extreme customer

In [ ]:
X_out = np.vstack([X2, [[400, 99]]])                       # one extra customer: Income 400, Spending 99
km_a = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2)
km_b = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X_out)
def summary(km):
    o = np.argsort(km.cluster_centers_[:, 0]); return pd.DataFrame({'Size': np.bincount(km.labels_)[o], 'Income': km.cluster_centers_[o, 0], 'Spending': km.cluster_centers_[o, 1]}).round(1)
print('WITHOUT the extra customer (sorted by income):'); display(summary(km_a))
print('WITH the extra customer:'); display(summary(km_b))
ari_out = adjusted_rand_score(km_a.labels_, km_b.labels_[:200]); print('ARI between the two results on the original 200 customers:', round(ari_out, 4))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, km, Xp, t in zip(ax, [km_a, km_b], [X2, X_out], ['Without outlier', 'With outlier (Income=400, Spending=99)']):
    for k in range(5): a.scatter(*Xp[km.labels_ == k].T, s=22, color=PAL[k], alpha=.8)
    a.scatter(*km.cluster_centers_.T, marker='X', s=220, c='black', edgecolor='white'); a.set_title(t); a.set_xlabel('Income'); a.set_ylabel('Spending')
plt.tight_layout(); plt.show()
own = np.bincount(km_b.labels_)[km_b.labels_[-1]] == 1
say(f'''**Your numbers:** ARI (original 200 customers, with vs without the extra one) = **{ari_out:.3f}**. The extra customer {'was given a cluster **of its own** (size 1)' if own else 'was absorbed into a cluster and dragged its centroid'};
sizes without: {summary(km_a)['Size'].tolist()}, with: {summary(km_b)['Size'].tolist()} (both sorted by income).''')

### Task 5.2 — Catch it and repair

In [ ]:
d_real = km_a.transform(X2).min(axis=1)                          # distance of each real customer to its nearest centroid
d_extra = km_a.transform([[400, 99]]).min(axis=1)[0]
thr = d_real.mean() + 3 * d_real.std()                           # mean + 3 std (computed on the 200 real customers)
flag = d_real > thr
print(f'Threshold = {d_real.mean():.2f} + 3 x {d_real.std():.2f} = {thr:.2f}')
print(f'Real customers flagged: {int(flag.sum())} of 200 | extra customer distance = {d_extra:.1f} -> flagged: {d_extra > thr}')
top5 = df.assign(Cluster=km_a.labels_, Distance=d_real.round(2)).sort_values('Distance', ascending=False).head(5)[['CustomerID', 'Age', 'Income', 'Spending', 'Cluster', 'Distance']]
display(top5)
keep = ~flag; km_fix = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2[keep])
ari_fix = adjusted_rand_score(km_a.labels_, km_fix.predict(X2))
print('Sizes after removing flagged points:', np.bincount(km_fix.labels_).tolist(), '| ARI vs original model on all 200 customers:', round(ari_fix, 4))
say(f'''**Your numbers:** threshold **{thr:.1f}**; the extra customer is **{d_extra:.0f}** away from its nearest centroid → **{'flagged' if d_extra > thr else 'not flagged'}**; **{int(flag.sum())}** real customers are flagged.
After removing the flagged points and refitting, ARI against the original 5-cluster model = **{ari_fix:.3f}**.''')

### Answers — Part 5
**What the single extreme customer did.** Compare the two tables above. K-Means minimises *squared* distances, and a point at Income = 400 (more than 250 k$ beyond the next-highest customer) has an enormous squared distance, so the algorithm re-allocates its limited K = 5 centroids to reduce that cost: typically the outlier gets a cluster **of its own** (size 1) and, because K is fixed, two of the real groups have to be **merged** (usually the two neighbouring groups, most often the high-income pair or the low-spending ones) or the real boundaries shift. The ARI on the original 200 customers (printed above) falls well below 1, i.e. the segmentation of real customers changed because of one extra row; if the outlier had instead been absorbed into a cluster, its centroid would have been dragged towards it, also shifting the boundaries.

**How well did the rule work?** The extra customer is thousands of times farther from its nearest centroid than a typical customer, so the mean + 3 std rule flags it (see the printed distance vs threshold). Check the printed count of flagged *real* customers. If the rule flags some (or when you read the 5 farthest customers listed with their Income and Spending), they are normally not data errors but genuine customers at the edge of their group — e.g. unusually high income with middling spending — who do not fit any segment well and are worth a look (possible VIPs or cross-segment customers) rather than deletion. If it flags none, the mean + 3 std rule is simply conservative on a small, compact data set, and the farthest-5 list is the better place to look.

**Two ways to deal with outliers, and one risk.** (1) Detect and remove (or separately handle) them before clustering, as done here (distance rule, or the IQR/z-score rule); (2) use a method or transformation that is robust to them — K-Medoids/DBSCAN, winsorising/capping extreme values, or a log transform. (Also: increase K deliberately and treat the singleton cluster as an "outlier group".) The risk of simply deleting is that the points may be the most interesting customers (high-value, fraud, data-entry error that reveals a process problem) and that removing them makes the model blind to such customers when it is used on new data, and the 3-std rule itself depends on the mean and std that the outliers inflate.

## Part 6 — From Clusters to Customer Segments

### Task 6.1 — Profile and name the segments

In [ ]:
km6 = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit(X2); df['Cluster'] = km6.labels_
profile = df.groupby('Cluster').agg(Customers=('CustomerID', 'count'), Age=('Age', 'mean'), Income=('Income', 'mean'), Spending=('Spending', 'mean'),
                                     Pct_female=('Gender', lambda s: 100 * (s == 'Female').mean())).sort_values('Income')
display(profile.round(1))

def level(v, lo, hi): return 'Low' if v < lo else ('High' if v >= hi else 'Mid')
names, seen = {}, {}
for c, r in profile.iterrows():
    nm = f"{level(r['Income'], 40, 70)} Income / {level(r['Spending'], 35, 65)} Spending"
    seen[nm] = seen.get(nm, 0) + 1; names[c] = nm if seen[nm] == 1 else f'{nm} (#{seen[nm]})'
df['Segment'] = df['Cluster'].map(names)
display(df['Segment'].value_counts().rename('Customers').to_frame())

### Task 6.2 — Does Gender matter? Does Age matter?

In [ ]:
from scipy.stats import chi2_contingency
overall_f = 100 * (df['Gender'] == 'Female').mean()
gtab = df.groupby('Segment')['Gender'].apply(lambda s: 100 * (s == 'Female').mean()).round(1).rename('% female').to_frame(); gtab.loc['ALL customers'] = round(overall_f, 1); display(gtab)
print(pd.crosstab(df['Segment'], df['Gender'], normalize='index').round(3))
chi_p = chi2_contingency(pd.crosstab(df['Segment'], df['Gender']))[1]; print(f'Chi-square test of independence (Segment vs Gender): p = {chi_p:.3f}')

X3 = StandardScaler().fit_transform(df[['Age', 'Income', 'Spending']].values.astype(float))
s3 = {k: silhouette_score(X3, KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X3)) for k in range(2, 9)}
s3t = pd.Series(s3, name='Silhouette (scaled Age, Income, Spending)').rename_axis('K').to_frame(); display(s3t.round(4))
best3 = int(s3t['Silhouette (scaled Age, Income, Spending)'].idxmax()); print('Best K with 3 features:', best3)

### Task 6.3 — A three-feature alternative

In [ ]:
km3 = KMeans(n_clusters=best3, n_init=10, random_state=SEED).fit(X3); df['Cluster3'] = km3.labels_
prof3 = df.groupby('Cluster3').agg(Customers=('CustomerID', 'count'), Age=('Age', 'mean'), Income=('Income', 'mean'), Spending=('Spending', 'mean')).sort_values('Age'); display(prof3.round(1))
ari_23 = adjusted_rand_score(df['Cluster'], df['Cluster3']); print(f'ARI between the 3-feature (K={best3}) and 2-feature (K=5) segmentations: {ari_23:.3f}')

In [ ]:
ACTIONS = {('High','High'): ('VIP / most valuable', 'loyalty tier, early access to new collections, premium perks', 'loyalty tier + early access'),
           ('High','Mid'):  ('Upsell candidates', 'personalised premium recommendations and bundles', 'premium bundles'),
           ('High','Low'):  ('Untapped wealth (biggest opportunity)', 'targeted premium offers, personal outreach and a survey to learn why they spend little here', 'premium offers + outreach'),
           ('Mid','High'):  ('Engaged spenders', 'reward programmes and cross-sell to keep spending', 'rewards + cross-sell'),
           ('Mid','Mid'):   ('Core mainstream', 'standard seasonal promotions, bundle discounts and loyalty points to nudge them up', 'seasonal promos + points'),
           ('Mid','Low'):   ('Cautious middle', 'value-driven coupons and reminders', 'value coupons'),
           ('Low','High'):  ('Enthusiastic budget shoppers', 'discounts, instalment or student offers, points programme (watch margins)', 'discounts + instalments'),
           ('Low','Mid'):   ('Budget regulars', 'low-cost promotions and clearance alerts', 'clearance alerts'),
           ('Low','Low'):   ('Low-engagement / low-value', 'cheap channels only (email/SMS clearance alerts); avoid heavy spending on them', 'low-cost email/SMS only')}
act_rows = []
for c, r in profile.iterrows():
    key = (level(r['Income'], 40, 70), level(r['Spending'], 35, 65)); nick, action, short = ACTIONS[key]
    act_rows.append({'Segment': names[c], 'Customers': int(r['Customers']), 'Avg income (k$)': round(r['Income'], 1), 'Avg spending': round(r['Spending'], 1),
                     'Business name': nick, 'Action': action, 'Short': short, 'Value proxy': r['Customers'] * r['Income'] * r['Spending'] / 100,
                     'Opportunity proxy': r['Customers'] * r['Income'] * (100 - r['Spending']) / 100})
acts = pd.DataFrame(act_rows).set_index('Segment'); pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 250)
display(acts[['Customers', 'Avg income (k$)', 'Avg spending', 'Business name', 'Action']])
most_val = acts['Value proxy'].idxmax(); opp = acts['Opportunity proxy'].idxmax()
fmax, fmin = gtab.drop('ALL customers')['% female'].max(), gtab.drop('ALL customers')['% female'].min()
say(f'''**Your numbers:** most valuable now (proxy = customers × income × spending score): **{most_val}**; biggest opportunity (proxy = customers × income × unused spending score): **{opp}**.
Overall {overall_f:.1f}% of customers are female; per segment it ranges from **{fmin:.0f}% to {fmax:.0f}%** (a spread of {fmax-fmin:.0f} points); chi-square test of Segment vs Gender: p = **{chi_p:.3f}** → gender and segment are {'**not** significantly related at the 5% level' if chi_p >= 0.05 else 'significantly related at the 5% level'}.
3-feature silhouette is highest at K = **{best3}**; ARI between the 3-feature and the 2-feature segmentation = **{ari_23:.3f}**.''')

### Answers — Part 6
**Segment names and actions.** The table above gives, for each of the 5 clusters, its size, average income and spending, a business name and one concrete action — taken from the profile numbers (for instance *High Income / High Spending → VIP*: loyalty tier and early access; *High Income / Low Spending → Untapped wealth*: personal outreach and premium offers; *Low Income / High Spending → Enthusiastic budget shoppers*: discounts and instalment offers while watching margins; *Mid / Mid → Core mainstream*: standard promotions and points; *Low / Low → Low-engagement*: cheap channels only). Quote the exact sizes and averages from your own table when you submit.

**Most valuable vs biggest opportunity.** The *most valuable now* segment is the one that combines a large number of customers with high income and a high spending score — normally **High Income / High Spending** (or the large mainstream segment if its size outweighs the smaller VIP group; the proxy in the generated sentence settles this, but it is only a rough proxy, not revenue). The *biggest opportunity* is the segment with **high income but low spending**: these customers have the means but spend little here, so each point of extra engagement is worth the most. This is where marketing effort has the highest upside.

**Does Gender help?** Look at the % female per segment against the overall percentage (printed above): compare the spread between segments with the overall share and with the chi-square p-value in the generated sentence. With only 200 customers and segments of 25–80 people, differences of 10–20 points can appear by chance, so unless the test is significant **Gender does not separate the segments** and would not be a useful clustering feature; it is best used as a descriptive variable, not a driver. (Quote the actual percentages from your table.)

**Does Age change the story?** Age was not used in the K = 5 clustering. The 3-feature segmentation adds an age dimension (typically splitting spending by young vs older customers — the young carry high spending scores). The ARI printed above says how much the two segmentations agree: a high ARI means Age mostly re-draws the same groups; a low ARI means the 3-feature segmentation genuinely reorganises the customers. I would **present the 2-feature K = 5 segmentation** because its groups are clearly separated, easy to explain with a scatter plot and map directly to income/spending actions, and use the Age profile as supporting detail (e.g. which segments skew young); I would switch only if the 3-feature clusters clearly led to different actions. Silhouettes from the two feature sets cannot be compared directly because they are measured in different spaces.

## Part 7 — Final Output and Wrap-Up

### Task 7.1 — Save the segments

In [ ]:
out = df[['CustomerID', 'Age', 'Gender', 'Income', 'Spending', 'Cluster', 'Segment']]
out.to_csv('customer_segments.csv', index=False)
print('Saved customer_segments.csv with', len(out), 'rows'); display(out['Segment'].value_counts().rename('Customers').to_frame()); display(out.head())
try:
    from google.colab import files; files.download('customer_segments.csv')
except Exception: pass

In [ ]:
seg_list = '; '.join(f"**{n}** ({int(r['Customers'])}): {r['Short']}" for n, r in acts.iterrows())
say(f'''### Executive Summary
I chose **K = 5** for the Mall customers using Income and Spending: the elbow flattens after K = 5 (inertia falls {100*drop[5]/kt.loc[4,'Inertia']:.0f}% from K=4→5 but only {100*drop[6]/kt.loc[5,'Inertia']:.0f}% from K=5→6) and the average silhouette is highest at K = {bestK} ({kt.loc[bestK,'Silhouette']:.2f}).
The segments (size): {seg_list}.
Starting points matter: random starts reached the best inertia in only {r0['% runs reaching best']:.0f}% of 30 runs, so I use k-means++ with n_init=10 ({r3['% runs reaching best']:.0f}% of runs).
Features were standardised whenever units differ (changing Income from k$ to dollars changed the raw clustering, ARI {ari_t.iloc[0,0]:.2f}; on Wine scaling raised ARI from {wres.iloc[0,0]:.2f} to {wres.iloc[1,0]:.2f}); the two Mall features share a similar 0–140 scale so I clustered them as they are.
A distance rule (mean + 3 std) {'caught' if d_extra > thr else 'missed'} the artificial extreme customer and flagged {int(flag.sum())} real customers{', which I reviewed rather than deleted' if flag.sum() > 0 else ''}.
Gender {'did not separate' if chi_p >= 0.05 else 'partly separated'} the segments ({fmin:.0f}%–{fmax:.0f}% female vs {overall_f:.0f}% overall, chi-square p = {chi_p:.2f}) and adding Age gave a different grouping (ARI {ari_23:.2f}).
**Limitation:** only 200 customers, two behavioural features and no revenue data, so "value" is a proxy and the segments are a snapshot that should be re-validated on more data.''')

### Final Reflection
**(a) Most surprising result:** *(write your own — candidates from this lab: how much changing Income from k$ to dollars changed the clusters; how often random starts landed in a worse local minimum; that the raw Wine clustering had the higher silhouette but the worse ARI; or how one extra row rearranged the real segments.)*

**(b) With one more hour I would try:** a Gaussian Mixture or hierarchical clustering to compare with K-Means; bootstrapping the customers to measure how stable the 5 segments are; adding real revenue/visit-frequency features so "value" is not a proxy; K-Medoids or DBSCAN for robustness to outliers; and a PCA/UMAP plot of Age–Income–Spending to present the 3-feature view.